In [71]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
import duckdb
import pandas as pd
from src.possession_parser import Possession_parser

In [2]:
con = duckdb.connect(database='data/nba.sqlite', read_only=False)
# df = con.query("show tables").fetchdf()
# df

In [ ]:
## get game-ids

# df = con.query("select * from play_by_play limit 2").fetchdf()
# df

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600012,0,12,0,1,14:43 PM,12:00,None,Start of 1st Period (14:43 PM EST),None,...,None,None,0.0,0,None,None,None,None,None,0
1,0029600012,2,10,0,1,14:50 PM,12:00,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None,...,Suns,PHX,5.0,208,Sam Cassell,1610612756.0,Phoenix,Suns,PHX,0


In [ ]:
# df = con.query("select * from play_by_play where game_id = '0029600012'").fetchdf()

In [3]:
game_ids_df = con.query("select distinct game_id from play_by_play").fetchdf()
game_ids = game_ids_df['game_id']

In [57]:
cur_game_id = game_ids[0]
game_df = con.query(f"select * from play_by_play where game_id = '{cur_game_id}'").fetchdf()
game_df = game_df.sort_values(by='eventnum')
game_df

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_nickname,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag
0,0029600064,1,10,0,1,11:40 PM,12:00,Jump Ball Geiger vs. Baker: Tip to Robinson,None,None,...,Bucks,MIL,5.0,299,Glenn Robinson,1610612749.0,Milwaukee,Bucks,MIL,0
1,0029600064,2,2,5,1,11:41 PM,11:45,None,None,MISS Baker Layup,...,None,None,0.0,0,None,None,None,None,None,0
2,0029600064,3,4,0,1,11:41 PM,11:45,None,None,Bucks Rebound,...,None,None,0.0,0,None,None,None,None,None,0
3,0029600064,4,2,5,1,11:41 PM,11:41,None,None,MISS Allen Layup,...,None,None,0.0,0,None,None,None,None,None,0
4,0029600064,5,4,0,1,11:41 PM,11:40,Mason REBOUND (Off:0 Def:1),None,None,...,None,None,0.0,0,None,None,None,None,None,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,0029600064,444,5,1,4,13:59 PM,0:08,Geiger STEAL (1 STL),None,Robinson Bad Pass Turnover (P4.T12),...,Hornets,CHH,0.0,0,None,None,None,None,None,0
442,0029600064,445,2,1,4,13:59 PM,0:01,MISS Rice 3PT Jump Shot,None,None,...,None,None,0.0,0,None,None,None,None,None,0
443,0029600064,446,4,0,4,13:59 PM,0:00,Mason REBOUND (Off:3 Def:9),None,None,...,None,None,0.0,0,None,None,None,None,None,0
444,0029600064,447,9,1,4,13:59 PM,0:00,HORNETS Timeout: Regular (Full 7 Short 2),None,None,...,None,None,0.0,0,None,None,None,None,None,0


In [96]:
print(game_df.iloc[123])

game_id                                            0029600064
eventnum                                                  124
eventmsgtype                                                5
eventmsgactiontype                                          2
period                                                      2
wctimestring                                         12:12 PM
pctimestring                                             9:50
homedescription                           Curry STEAL (1 STL)
neutraldescription                                       None
visitordescription           Baker Lost Ball Turnover (P1.T4)
score                                                    None
scoremargin                                              None
person1type                                               5.0
player1_id                                                452
player1_name                                        Vin Baker
player1_team_id                                  1610612749.0
player1_

In [83]:
parser = Possession_parser(game_df)
possessions = parser.parse_game()
possessions

timeout
player not on offense scored at row 45
player not on offense scored at row 61
subbing: 5 players on team0
player not on offense scored at row 68
player not on offense scored at row 69
player not on offense scored at row 70
timeout
subbing: 5 players on team0
subbing: 5 players on team0
subbing: 5 players on team0
subbing: 5 players on team1
player not on offense scored at row 95
player not on offense scored at row 96
subbing: 5 players on team1
end of period
subbing: 5 players on team1
timeout
subbing: 5 players on team1
subbing: 5 players on team0
subbing: 5 players on team1
subbing: 5 players on team0
too many players on team1 @ row 129
too many players on team1 @ row 130
too many players on team1 @ row 131
too many players on team1 @ row 132
too many players on team1 @ row 133
too many players on team1 @ row 134
too many players on team1 @ row 135
too many players on team1 @ row 136
too many players on team1 @ row 137
too many players on team1 @ row 138
too many players on t

,game_id,possession_id,offense_team_id,possession_end_event_type,possession_end_points
0,0029600064,0,1610612749.0,4.0,0
1,0029600064,1,1610612766.0,1.5,2
2,0029600064,2,1610612749.0,4.0,0
3,0029600064,3,1610612766.0,4.0,0
4,0029600064,4,1610612749.0,1.1,2
...,...,...,...,...,...
423,0029600064,163,1610612766.0,1.1,2
424,0029600064,164,1610612749.0,5.1,0
425,0029600064,165,1610612766.0,5.1,0
426,0029600064,166,1610612749.0,4.0,0


In [ ]:
game_df

,game_id,eventnum,eventmsgtype,eventmsgactiontype,period,wctimestring,pctimestring,homedescription,neutraldescription,visitordescription,...,player2_team_abbreviation,person3type,player3_id,player3_name,player3_team_id,player3_team_city,player3_team_nickname,player3_team_abbreviation,video_available_flag,possession_index
0,0029600310,1,10,0,1,11:39 PM,12:00,Jump Ball Mutombo vs. Williams: Tip to Laettner,None,None,...,PHI,4.0,363,Christian Laettner,1610612737.0,Atlanta,Hawks,ATL,0,0029600310_0
1,0029600310,2,6,2,1,11:40 PM,11:55,None,None,Weatherspoon S.FOUL (P1.T1),...,PHI,0.0,0,None,None,None,None,None,0,0029600310_0
2,0029600310,3,3,11,1,11:40 PM,11:55,MISS Smith Free Throw 1 of 2,None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_0
3,0029600310,4,4,0,1,11:40 PM,11:55,HAWKS Rebound,None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_0
4,0029600310,5,3,12,1,11:40 PM,11:55,Smith Free Throw 2 of 2 (1 PTS),None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432,0029600310,442,1,5,4,13:41 PM,0:16,None,None,Bradtke Layup (4 PTS) (Harris 1 AST),...,PHI,0.0,0,None,None,None,None,None,0,0029600310_3
433,0029600310,443,5,2,4,13:41 PM,0:10,Barry Lost Ball Turnover (P2.T11),None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_3
434,0029600310,445,2,1,4,13:41 PM,0:02,None,None,MISS Davis 25' 3PT Jump Shot,...,None,0.0,0,None,None,None,None,None,0,0029600310_3
435,0029600310,446,4,0,4,13:41 PM,0:00,Recasner REBOUND (Off:0 Def:4),None,None,...,None,0.0,0,None,None,None,None,None,0,0029600310_3


In [ ]:
df[["homedescription",	"neutraldescription"	,"visitordescription"]]

,homedescription,neutraldescription,visitordescription
0,None,Start of 1st Period (14:43 PM EST),None
1,Jump Ball O'Neal vs. Kleine: Tip to Cassell,None,None
2,None,None,MISS Cassell 15' Jump Shot
3,O'Neal REBOUND (Off:0 Def:1),None,None
4,MISS Ceballos 26' 3PT Jump Shot,None,None
...,...,...,...
461,None,None,MISS Manning Free Throw 1 of 2
462,None,None,Suns Rebound
463,None,None,MISS Manning Free Throw 2 of 2
464,Knight REBOUND (Off:0 Def:2),None,None


In [ ]:
df.columns

Index(['game_id', 'eventnum', 'eventmsgtype', 'eventmsgactiontype', 'period',
       'wctimestring', 'pctimestring', 'homedescription', 'neutraldescription',
       'visitordescription', 'score', 'scoremargin', 'person1type',
       'player1_id', 'player1_name', 'player1_team_id', 'player1_team_city',
       'player1_team_nickname', 'player1_team_abbreviation', 'person2type',
       'player2_id', 'player2_name', 'player2_team_id', 'player2_team_city',
       'player2_team_nickname', 'player2_team_abbreviation', 'person3type',
       'player3_id', 'player3_name', 'player3_team_id', 'player3_team_city',
       'player3_team_nickname', 'player3_team_abbreviation',
       'video_available_flag'],
      dtype='object')

In [ ]:

## Pseudo logic for aggregation of player counting stats


# from collections import Counter

# players = Counter()

# for game in games:
#     for event in game.events:
#         player = event.player
#         stat = event.stat
#         players[player[stat]] += 1 

